#### Relevant Imports

# Notebook 1 — Connecting to MySQL & Multi-Table Queries

## What You Will Learn

This notebook introduces how to connect to a remote relational database from Python and retrieve data spanning multiple tables.

### Topics Covered

| Topic | Description |
|---|---|
| **SQLAlchemy Engine** | Database-agnostic abstraction that manages connections to any SQL backend |
| **Multiple Connections** | Creating and using several concurrent connections from a single engine |
| **SQL Queries** | Executing raw SQL via `text()` and iterating over results |
| **JOIN Operations** | Consolidating data across 3+ tables using foreign-key relationships |

### Skills You Will Build

- Connect to a remote MySQL database (RFAM — RNA family database) using `SQLAlchemy`
- Understand the `Engine → Connection → Query` lifecycle
- Write `JOIN` queries to pull coherent records from multiple tables
- Inspect row objects and fetch result data programmatically

> **Pre-requisite:** Basic SQL knowledge (SELECT, WHERE, JOIN). No prior SQLAlchemy experience needed.

In [ ]:
from sqlalchemy import create_engine, text

**Engine**  
Engine is the abstract object from SQLAlchmey which connects to a host to particular Database  
In an an engine further connections can be created to query data  
This generic concepts make it Database agnostic

In [ ]:
# There is an engine instance created, which can handle multiple connetions
sql_engine = create_engine("mysql+pymysql://rfamro:@mysql-rfam-public.ebi.ac.uk:4497/Rfam")

**Connection and Query**
Connection object creates a connection (its within the object)  
Then it is used to raise query (Execute Query String)

In [ ]:
# Create a Connection in engine and raise a query to get data
conn_1 = sql_engine.connect ()
result = conn_1.execute(text("SELECT COUNT(*) FROM full_region;"))

# Fetch the result content
for row in result:

    print (row)
    print (type(row))



In [ ]:
# Get more data, parallelly from another connection in same engine
conn_2 = sql_engine.connect ()
result = conn_2.execute(text("SELECT * FROM full_region LIMIT 10;"))

# Fetch the result content
print (result.keys())
for row in result:

    print (row)
    print (type(row))


In [ ]:
# For a Given value of rfam_acc, how many Ncbi Id
result = conn_1.execute (text(
                             "SELECT COUNT(DISTINCT fn.ncbi_id) AS ncbi_count\
                            FROM family_ncbi fn\
                            WHERE fn.rfam_acc = 'RF01530';"))

for row in result:

    print (row)    


**Data from multiple tables**  
Data from multiple table can be collated by the linking fields by JOIN queries

In [ ]:
# For a Given value of rfam_acc, how many species are present in another table based on its ncbi_id
result = conn_1.execute (text(
                             "SELECT DISTINCT fn.rfam_acc, fn.ncbi_id, t.species AS species\
                            FROM family_ncbi fn\
                            JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                            WHERE fn.rfam_acc = 'RF01530'"))

for row in result:

    print (row)    


In [ ]:
# Consolidated information from 3 tables
result = conn_1.execute (text(
                                "SELECT fn.rfam_id, fn.ncbi_id, t.species, f.rfam_id AS family_rfam_id, f.auto_wiki, f.description\
                                FROM family_ncbi fn\
                                JOIN taxonomy t ON fn.ncbi_id = t.ncbi_id\
                                JOIN family f ON fn.rfam_acc = f.rfam_acc\
                                WHERE fn.rfam_acc = 'RF01530'"))

rows = result.fetchall ()
print (rows)
  